# Notebook 3: Proposed FP-Growth với In-mining Pruning

Notebook này thực hiện kịch bản **Proposed (In-mining Pruning)** cho bài thực nghiệm khai phá luật kết hợp định lượng.

Ý tưởng chính:

1. Nạp danh sách giao dịch và bảng giá item từ `data_preprocessing.ipynb`.
2. Sắp xếp item theo **giá tăng dần**.
3. Xây FP-Tree theo thứ tự giá.
4. Duyệt sinh mẫu theo DFS và kiểm tra `avg(price)` ngay trong quá trình mở rộng mẫu.
5. Nếu `avg(price) > PRICE_THRESHOLD`, dừng mở rộng nhánh đó ngay.

Điểm quan trọng là DFS luôn mở rộng itemset theo thứ tự giá không giảm. Vì vậy, khi một ứng viên đã có `avg(price)` vượt ngưỡng, mọi mở rộng sau đó chỉ thêm item có giá bằng hoặc cao hơn, nên trung bình không thể giảm xuống dưới ngưỡng.

## 1. Cấu hình môi trường

Notebook chạy được trên local và Google Colab. Các tham số có thể chỉnh trực tiếp trong cell cấu hình hoặc bằng biến môi trường:

- `OUTPUT_DIR`: thư mục chứa artifact đầu vào và đầu ra.
- `MIN_SUPPORT_RATIOS`: danh sách support, ví dụ `5%,4%,3%,2%,1%`.
- `PRICE_THRESHOLD`: ngưỡng `avg(price)`, mặc định `20`.
- Các biến tên file artifact như `TRANSACTIONS_PICKLE`, `ITEM_PRICE_PICKLE`, `PROPOSED_RESULTS_PICKLE`.

In [1]:
import json
import math
import os
import pickle
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
root_dir = None
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists():
        root_dir = candidate
        break
if root_dir is None:
    root_dir = PROJECT_ROOT
# Ensure repo root for data/output paths when running from notebooks.
PROJECT_ROOT = root_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import (
    build_item_tid_bitsets,
    ensure_package,
    find_artifact,
    parse_support_ratios,
    patterns_to_dataframe,
    run_proposed_in_mining_pruning,
 )

ensure_package("pandas")

import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# =========================
# CẤU HÌNH LINH HOẠT
# =========================
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "outputs")).resolve()
DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data")).resolve()

ARTIFACT_FILES = {
    "transactions_pickle": os.environ.get("TRANSACTIONS_PICKLE", "transactions.pkl"),
    "item_price_pickle": os.environ.get("ITEM_PRICE_PICKLE", "item_price.pkl"),
    "preprocessing_summary_json": os.environ.get("PREPROCESSING_SUMMARY_JSON", "preprocessing_summary.json"),
    "baseline_results_pickle": os.environ.get("BASELINE_RESULTS_PICKLE", "baseline_results.pkl"),
    "proposed_results_pickle": os.environ.get("PROPOSED_RESULTS_PICKLE", "proposed_results.pkl"),
    "proposed_metrics_csv": os.environ.get("PROPOSED_METRICS_CSV", "proposed_metrics.csv"),
    "proposed_patterns_csv": os.environ.get("PROPOSED_PATTERNS_CSV", "proposed_final_patterns.csv"),
    "proposed_summary_json": os.environ.get("PROPOSED_SUMMARY_JSON", "proposed_summary.json"),
}

MIN_SUPPORT_RATIOS = parse_support_ratios(
    os.environ.get("MIN_SUPPORT_RATIOS", ""),
    default_ratios=[0.05, 0.04, 0.03, 0.02, 0.01],
)
PRICE_THRESHOLD = float(os.environ.get("PRICE_THRESHOLD", "20"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 3000)

print("Môi trường chạy:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Thư mục làm việc:", PROJECT_ROOT)
print("Thư mục dữ liệu:", DATA_DIR)
print("Thư mục lưu kết quả:", OUTPUT_DIR)
print("Danh sách min_support:", [f"{ratio:.0%}" for ratio in MIN_SUPPORT_RATIOS])
print("Ngưỡng avg(price):", PRICE_THRESHOLD)
if not DATA_DIR.exists():
    print("Warning: DATA_DIR not found:", DATA_DIR)

Môi trường chạy: Local/Jupyter
Thư mục làm việc: D:\Projects\Constraint-based_pattern_data_mining
Thư mục dữ liệu: D:\Projects\Constraint-based_pattern_data_mining\data
Thư mục lưu kết quả: D:\Projects\Constraint-based_pattern_data_mining\outputs
Danh sách min_support: ['5%', '4%', '3%', '2%', '1%']
Ngưỡng avg(price): 20.0


## 2. Nạp artifact từ bước tiền xử lý

Notebook sử dụng các artifact đã tạo ở notebook 1:

- Danh sách giao dịch dạng pickle.
- Bảng giá đại diện của từng item.
- File tóm tắt tiền xử lý.

Nếu chạy trên Colab, hãy chạy notebook 1 trước hoặc upload các file artifact vào cùng thư mục làm việc.

In [2]:
transactions_path = find_artifact(
    ARTIFACT_FILES["transactions_pickle"],
    output_dir=OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
)
item_price_path = find_artifact(
    ARTIFACT_FILES["item_price_pickle"],
    output_dir=OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
)
summary_path = find_artifact(
    ARTIFACT_FILES["preprocessing_summary_json"],
    output_dir=OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
)

required_artifacts = [transactions_path, item_price_path, summary_path]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(
        "Thiếu artifact từ bước tiền xử lý. Hãy chạy data_preprocessing.ipynb trước. "
        f"Các file còn thiếu: {missing_artifacts}"
    )

with transactions_path.open("rb") as f:
    transactions = pickle.load(f)

with item_price_path.open("rb") as f:
    item_price = pickle.load(f)

with summary_path.open("r", encoding="utf-8") as f:
    preprocessing_summary = json.load(f)

item_price = {item: float(price) for item, price in item_price.items()}
N_TRANSACTIONS = len(transactions)

load_summary_df = pd.DataFrame([
    {"Thông tin": "Số giao dịch", "Giá trị": N_TRANSACTIONS},
    {"Thông tin": "Số item có giá đại diện", "Giá trị": len(item_price)},
    {"Thông tin": "Số dòng sau tiền xử lý", "Giá trị": preprocessing_summary.get("clean_rows")},
    {"Thông tin": "Cách lấy giá đại diện", "Giá trị": preprocessing_summary.get("price_agg_method", "median")},
])

print("Đã nạp dữ liệu tiền xử lý thành công.")
display(load_summary_df)

print("Ví dụ 5 giao dịch đầu:")
display(pd.DataFrame({
    "Mã thứ tự giao dịch": list(range(1, 6)),
    "Số item": [len(t) for t in transactions[:5]],
    "Items": [" | ".join(t) for t in transactions[:5]],
}))

Đã nạp dữ liệu tiền xử lý thành công.


,Thông tin,Giá trị
0,Số giao dịch,19559
1,Số item có giá đại diện,4006
2,Số dòng sau tiền xử lý,519551
3,Cách lấy giá đại diện,median


Ví dụ 5 giao dịch đầu:


,Mã thứ tự giao dịch,Số item,Items
0,1,7,CREAM CUPID HEARTS COAT HANGER | GLASS STAR FROSTED T-LIGHT HOLDER | KNITTED UNION FLAG HOT WATER BOTTLE | RED WOOLLY HOTTIE WHITE HEART. | SET 7 BABUSHKA NESTING BOXES | WHITE HANGING HEART T-LIG...
1,2,2,HAND WARMER RED POLKA DOT | HAND WARMER UNION JACK
2,3,12,ASSORTED COLOUR BIRD ORNAMENT | BOX OF 6 ASSORTED COLOUR TEASPOONS | BOX OF VINTAGE ALPHABET BLOCKS | BOX OF VINTAGE JIGSAW BLOCKS | DOORMAT NEW ENGLAND | FELTCRAFT PRINCESS CHARLOTTE DOLL | HOME ...
3,4,4,BLUE COAT RACK PARIS FASHION | JAM MAKING SET WITH JARS | RED COAT RACK PARIS FASHION | YELLOW COAT RACK PARIS FASHION
4,5,1,BATH BUILDING BLOCK WORD


## 3. Cấu hình thực nghiệm Proposed

Các ngưỡng support được đổi sang support count bằng `ceil(min_support_ratio * số_giao_dịch)`. Ngưỡng định lượng mặc định là `avg(price) <= 20`.

Khác với Baseline, Proposed không sinh toàn bộ frequent itemsets rồi mới lọc. Mỗi candidate được kiểm tra giá ngay khi đang được sinh trong DFS.

In [3]:
support_config_df = pd.DataFrame([
    {
        "Min Support": f"{ratio:.0%}",
        "Tỷ lệ": ratio,
        "Support count tối thiểu": math.ceil(ratio * N_TRANSACTIONS),
    }
    for ratio in MIN_SUPPORT_RATIOS
])

print("Cấu hình thực nghiệm Proposed:")
display(support_config_df)
print(f"Ngưỡng cắt tỉa trong quá trình mining: avg(price) <= {PRICE_THRESHOLD}")

Cấu hình thực nghiệm Proposed:


,Min Support,Tỷ lệ,Support count tối thiểu
0,5%,0.05,978
1,4%,0.04,783
2,3%,0.03,587
3,2%,0.02,392
4,1%,0.01,196


Ngưỡng cắt tỉa trong quá trình mining: avg(price) <= 20.0


## 4. Cài đặt FP-Tree theo thứ tự giá và DFS pruning

Cài đặt gồm hai phần:

- **Price-ordered FP-Tree:** mỗi giao dịch được lọc item thường xuyên rồi sắp theo giá tăng dần trước khi đưa vào cây.
- **DFS in-mining pruning:** itemset được mở rộng theo đúng thứ tự giá tăng dần. Nếu `avg(price)` của candidate vượt ngưỡng, notebook dừng phần còn lại của nhánh hiện tại.

Để tính support nhanh trong DFS, notebook tạo chỉ mục TID dạng bitset cho từng item. Đây là header index dùng chung cho các ngưỡng support, giúp thao tác giao tập giao dịch được thực hiện bằng phép toán bit nhanh thay vì quét lại toàn bộ dữ liệu nhiều lần.

In [4]:
index_start_time = time.perf_counter()
item_bitsets = build_item_tid_bitsets(transactions, item_price)
index_build_seconds = time.perf_counter() - index_start_time

print("Đã định nghĩa xong Proposed FP-Tree và DFS in-mining pruning.")
print(f"Thời gian tạo chỉ mục TID bitset dùng chung: {index_build_seconds:.6f} giây")
print("Số item có bitset giao dịch:", len(item_bitsets))

Đã định nghĩa xong Proposed FP-Tree và DFS in-mining pruning.
Thời gian tạo chỉ mục TID bitset dùng chung: 0.367523 giây
Số item có bitset giao dịch: 4006


## 5. Chạy thực nghiệm Proposed

Mỗi ngưỡng support được chạy riêng. Thời gian Proposed trong bảng dưới đây bao gồm bước lọc frequent items, xây FP-Tree theo giá và DFS sinh mẫu có pruning. Chỉ mục TID bitset được tạo một lần ở trên và được ghi riêng để minh bạch.

In [5]:
proposed_results = {}
metrics_rows = []

for ratio in MIN_SUPPORT_RATIOS:
    label = f"{ratio:.0%}"
    print(f"Đang chạy Proposed In-mining Pruning với min_support = {label}...")
    result = run_proposed_in_mining_pruning(
        transactions=transactions,
        item_price=item_price,
        item_bitsets=item_bitsets,
        min_support_ratio=ratio,
        price_threshold=PRICE_THRESHOLD,
    )
    proposed_results[label] = result
    stats = result["stats"]

    metrics_rows.append({
        "Min Support": label,
        "Support count tối thiểu": result["min_support_count"],
        "Thời gian Proposed (giây)": round(result["runtime_seconds"], 6),
        "Số lượng Final Patterns": result["num_final_patterns"],
        "Số item thường xuyên ở gốc": stats["root_frequent_items"],
        "Số node FP-Tree theo giá": stats["fp_tree_nodes"],
        "Số candidate bị cắt do avg(price)": stats["price_pruned_candidates"],
        "Số candidate bị loại do support": stats["infrequent_candidates"],
        "Số lần gọi DFS": stats["recursive_calls"],
        "Độ dài mẫu lớn nhất": stats["max_pattern_length"],
    })

    print(
        f"Hoàn tất {label}: "
        f"{result['num_final_patterns']} final patterns, "
        f"cắt {stats['price_pruned_candidates']} candidate do avg(price), "
        f"runtime = {result['runtime_seconds']:.4f} giây."
    )

proposed_metrics_df = pd.DataFrame(metrics_rows)

print("Bảng kết quả Proposed:")
display(proposed_metrics_df)

Đang chạy Proposed In-mining Pruning với min_support = 5%...
Hoàn tất 5%: 33 final patterns, cắt 0 candidate do avg(price), runtime = 0.0709 giây.
Đang chạy Proposed In-mining Pruning với min_support = 4%...
Hoàn tất 4%: 66 final patterns, cắt 0 candidate do avg(price), runtime = 0.1532 giây.
Đang chạy Proposed In-mining Pruning với min_support = 3%...
Hoàn tất 3%: 137 final patterns, cắt 138 candidate do avg(price), runtime = 0.3295 giây.
Đang chạy Proposed In-mining Pruning với min_support = 2%...
Hoàn tất 2%: 379 final patterns, cắt 380 candidate do avg(price), runtime = 0.6976 giây.
Đang chạy Proposed In-mining Pruning với min_support = 1%...
Hoàn tất 1%: 1862 final patterns, cắt 1863 candidate do avg(price), runtime = 2.6052 giây.
Bảng kết quả Proposed:


,Min Support,Support count tối thiểu,Thời gian Proposed (giây),Số lượng Final Patterns,Số item thường xuyên ở gốc,Số node FP-Tree theo giá,Số candidate bị cắt do avg(price),Số candidate bị loại do support,Số lần gọi DFS,Độ dài mẫu lớn nhất
0,5%,978,0.070857,33,33,13634,0,528,34,1
1,4%,783,0.153244,66,65,37250,0,2111,67,2
2,3%,587,0.329456,137,130,79783,138,8669,138,2
3,2%,392,0.697609,379,298,158945,380,56561,380,3
4,1%,196,2.605220,1862,812,300916,1863,697890,1863,5


## 6. Hiển thị đầy đủ các final patterns của Proposed

Bảng dưới đây hiển thị các itemset được sinh hợp lệ trong quá trình mining, tức các mẫu đã đạt `min_support` và `avg(price) <= PRICE_THRESHOLD` mà không cần bước hậu xử lý.

In [6]:
all_final_patterns_df = pd.concat(
    [patterns_to_dataframe(result, item_price, N_TRANSACTIONS) for result in proposed_results.values()],
    ignore_index=True,
)

all_final_patterns_df["Support"] = all_final_patterns_df["Support"].round(6)
all_final_patterns_df["Avg Price"] = all_final_patterns_df["Avg Price"].round(6)

# print("Toàn bộ final patterns của Proposed:")
# display(all_final_patterns_df)

## 7. Lưu kết quả Proposed

Các kết quả được lưu để notebook đánh giá có thể lập bảng so sánh runtime, số lượng final patterns và vẽ biểu đồ.

In [7]:
proposed_results_path = OUTPUT_DIR / ARTIFACT_FILES["proposed_results_pickle"]
proposed_metrics_path = OUTPUT_DIR / ARTIFACT_FILES["proposed_metrics_csv"]
proposed_patterns_path = OUTPUT_DIR / ARTIFACT_FILES["proposed_patterns_csv"]
proposed_summary_path = OUTPUT_DIR / ARTIFACT_FILES["proposed_summary_json"]

with proposed_results_path.open("wb") as f:
    pickle.dump(proposed_results, f)

proposed_metrics_df.to_csv(proposed_metrics_path, index=False, encoding="utf-8-sig")
all_final_patterns_df.to_csv(proposed_patterns_path, index=False, encoding="utf-8-sig")

proposed_summary = {
    "price_threshold": PRICE_THRESHOLD,
    "num_transactions": N_TRANSACTIONS,
    "index_build_seconds": index_build_seconds,
    "min_supports": [f"{ratio:.0%}" for ratio in MIN_SUPPORT_RATIOS],
    "metrics": proposed_metrics_df.to_dict(orient="records"),
}

with proposed_summary_path.open("w", encoding="utf-8") as f:
    json.dump(proposed_summary, f, ensure_ascii=False, indent=2)

saved_files_df = pd.DataFrame([
    {"File": str(proposed_results_path), "Mô tả": "Kết quả Proposed dạng pickle để evaluation dùng so sánh"},
    {"File": str(proposed_metrics_path), "Mô tả": "Bảng runtime và số lượng pattern của Proposed"},
    {"File": str(proposed_patterns_path), "Mô tả": "Toàn bộ final patterns của Proposed"},
    {"File": str(proposed_summary_path), "Mô tả": "Tóm tắt kết quả Proposed dạng JSON"},
])

print("Đã lưu xong kết quả Proposed:")
display(saved_files_df)

Đã lưu xong kết quả Proposed:


,File,Mô tả
0,D:\Projects\Constraint-based_pattern_data_mining\outputs\proposed_results.pkl,Kết quả Proposed dạng pickle để evaluation dùng so sánh
1,D:\Projects\Constraint-based_pattern_data_mining\outputs\proposed_metrics.csv,Bảng runtime và số lượng pattern của Proposed
2,D:\Projects\Constraint-based_pattern_data_mining\outputs\proposed_final_patterns.csv,Toàn bộ final patterns của Proposed
3,D:\Projects\Constraint-based_pattern_data_mining\outputs\proposed_summary.json,Tóm tắt kết quả Proposed dạng JSON
